In [ ]:
import duckdb
import os
cn = duckdb.connect()
sql = f"""
    CREATE SECRET gcs_secrets (TYPE GCS, KEY_ID '{os.getenv("GCS_KEY")}', SECRET '{os.getenv("GCS_SECRET")}');
    ATTACH 'ducklake:gcs_wh.ducklake' as gcs_wh (DATA_PATH 'gs://{os.getenv("GCS_BUCKET")}/ducklake');
"""
cn.execute(sql)

In [2]:
sql = """
    create or replace table gcs_wh.orders as
    select 1 as order_id, '2024-01-01'::date as order_date, 100.0 as amount
    union all
    select 2 as order_id, '2024-01-02'::date as order_date, 150.0 as amount
"""
cn.execute(sql)

In [3]:
cn.sql("select * from gcs_wh.orders").show()

┌──────────┬────────────┬──────────────┐
│ order_id │ order_date │    amount    │
│  int32   │    date    │ decimal(4,1) │
├──────────┼────────────┼──────────────┤
│        1 │ 2024-01-01 │        100.0 │
│        2 │ 2024-01-02 │        150.0 │
└──────────┴────────────┴──────────────┘



In [4]:
sql = """
    create or replace table gcs_wh.orders_tmp as
    select 1 as order_id, '2024-01-01'::date as order_date, 101.0 as amount
    union all
    select 2 as order_id, '2024-01-02'::date as order_date, 151.0 as amount
    union all
    select 3 as order_id, '2024-01-03'::date as order_date, 200.0 as amount
"""
cn.execute(sql)

In [9]:
sql = """
    MERGE INTO gcs_wh.orders AS target
        USING gcs_wh.orders_tmp AS source
            ON target.order_id = source.order_id
    WHEN MATCHED THEN UPDATE 
    WHEN NOT MATCHED THEN INSERT BY NAME
"""
cn.execute(sql)

In [8]:
cn.sql("select * from gcs_wh.orders").show()

┌──────────┬────────────┬──────────────┐
│ order_id │ order_date │    amount    │
│  int32   │    date    │ decimal(4,1) │
├──────────┼────────────┼──────────────┤
│        3 │ 2024-01-03 │        200.0 │
│        1 │ 2024-01-01 │        101.0 │
│        2 │ 2024-01-02 │        151.0 │
└──────────┴────────────┴──────────────┘



In [10]:
cn.close()